# 04 – Detecting deforestation at Hasdeo Arand

This is the research part. Everything so far was building and testing a tool; now we point it at the real world.

**Study area:** the **Parsa East and Kanta Basan (PEKB)** open-pit coal mine in the Hasdeo Arand forest, Chhattisgarh, India (about 22.838° N, 82.814° E). One of central India's largest dense forests, partly being cleared for coal mining.

**The plan:**
1. Search the free Sentinel-2 archive for cloud-free images from **February 2019** and **February 2025**
2. Download a 16 km × 16 km area around the mine, for both dates
3. Check how different these images look from EuroSAT (the *domain shift* you predicted)
4. Chop each image into 64 × 64 patches, exactly the size the model was trained on
5. Classify every patch → a **land cover map** for each year
6. Compare the maps: where did **Forest** become something else?

Runs in 1–2 minutes (it downloads ~15 MB). **Run → Run All Cells**, then read along.

**New libraries** (already installed):
- `pystac_client` searches satellite image catalogues (STAC = SpatioTemporal Asset Catalog, a standard way of listing satellite images)
- `rasterio` reads satellite image files and knows *where on Earth* each pixel is

## Step 1 – Setup

`LAT, LON` is the centre of our study area (the mine). `N_PATCHES = 25` means a grid of 25 × 25 patches. Each patch is 64 pixels × 10 m = **640 m**, so the whole area is 25 × 640 m = **16 km** on each side.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from PIL import Image

import rasterio
from rasterio.warp import transform as reproject_points
from rasterio.windows import Window
from pystac_client import Client

import torch
import torch.nn as nn
from torchvision import models, transforms

DATA_DIR = Path("C:/data/EuroSAT")
MODEL_PATH = Path("../models/resnet50_eurosat.pth")
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("../results"); RESULTS_DIR.mkdir(exist_ok=True)

LAT, LON = 22.838, 82.814      # PEKB coal mine, Hasdeo Arand
N_PATCHES = 25                  # 25 x 25 grid of 64x64 patches = 16 km x 16 km
PATCH = 64
PATCH_KM2 = (PATCH * 10 / 1000) ** 2   # 0.64 km x 0.64 km = 0.41 km² per patch

device = "cuda" if torch.cuda.is_available() else "cpu"
class_names = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print("Running on:", device, "| study area:", N_PATCHES * PATCH * 10 / 1000, "km per side")

## Step 2 – Search for satellite images

We use **Earth Search**, a free public catalogue of every Sentinel-2 image, run by Element 84 on Amazon's cloud. No account needed.

We ask for:
- the `sentinel-2-l2a` collection. **L2A** means the image has been corrected for the atmosphere (haze removed), so the colours are closer to the real ground
- images that cover our point
- taken between January and March (the **dry season**: fewer clouds, and both years in the same season, so seasonal change in the plants doesn't get mistaken for deforestation)
- less than 5% cloud cover

Each result shows its **ID**, date, cloud %, and its **tile** (Sentinel-2 cuts the Earth into fixed 110 km squares with codes like `44QPL`). Both dates must come from the same tile so the pixels line up perfectly.

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
point = {"type": "Point", "coordinates": [LON, LAT]}

for period in ["2019-01-01/2019-03-31", "2025-01-01/2025-03-31"]:
    results = catalog.search(collections=["sentinel-2-l2a"], intersects=point, datetime=period,
                             query={"eo:cloud_cover": {"lt": 5}})
    items = sorted(results.items(), key=lambda item: item.properties["datetime"])
    print(f"{period}: {len(items)} images with < 5% cloud")
    for item in items:
        p = item.properties
        print(f"   {item.id:28} {p['datetime'][:10]}  cloud {p['eo:cloud_cover']:5.2f}%  tile {p['grid:code']}")
    print()

Plenty of clear images. We pick one per year, **as close to the same date as possible** (20 Feb 2019 and 8 Feb 2025), so the forest is at the same stage of the season in both.

We write the IDs down explicitly instead of "take the first result", so anyone re-running this gets exactly the same images. That's what makes it **reproducible**, which a paper needs.

In [ ]:
SCENES = {
    2019: "S2A_44QPL_20190220_0_L2A",
    2025: "S2C_44QPL_20250208_0_L2A",
}

## Step 3 – Download our area from each image

A full Sentinel-2 tile is 10,980 × 10,980 pixels (~110 km). We only need a 1,600 × 1,600 pixel square around the mine.

Each image has a `visual` file: a ready-made true-colour picture (red, green, blue, 0–255) like a normal photo, made by ESA. It's stored as a **Cloud-Optimised GeoTIFF**, which means `rasterio` can download *just our square* instead of the whole 100+ MB file.

**Coordinates:** satellite tiles don't use latitude/longitude. They use **UTM**: metres east and north inside a zone (our tile uses UTM zone 44N). So we convert our lat/lon to UTM, then to a row/column inside the image, then cut a `Window` around it.

We also keep `geo`: where our square sits on Earth. Notebook 05 needs it to line our results up with Global Forest Watch's data.

In [ ]:
size = N_PATCHES * PATCH      # 1600 pixels
images, geo = {}, {}

for year, scene_id in SCENES.items():
    item = catalog.get_collection("sentinel-2-l2a").get_item(scene_id)
    with rasterio.open(item.assets["visual"].href) as src:
        xs, ys = reproject_points("EPSG:4326", src.crs, [LON], [LAT])
        row, col = src.index(xs[0], ys[0])
        window = Window(col - size // 2, row - size // 2, size, size)
        pixels = src.read(window=window)                 # shape: (3 colours, 1600, 1600)
        geo[year] = {"crs": src.crs.to_string(), "transform": tuple(src.window_transform(window))[:6]}
    images[year] = np.moveaxis(pixels, 0, -1)            # -> (1600, 1600, 3), like notebook 01
    print(f"{year}: {item.properties['datetime'][:10]}  image {images[year].shape}  CRS {geo[year]['crs']}")

assert geo[2019] == geo[2025], "the two images must line up exactly"
print("Both images line up pixel for pixel.")

### The two images, side by side

Look for:
- the **open-pit mine** in the centre (pale and grey stripes = exposed rock and coal)
- how much **bigger** it is in 2025
- a **new cleared area** to the north-west of the mine in 2025
- the dark brown-green areas are forest; the orange/tan areas are fields, villages, scrub and bare soil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, year in zip(axes, SCENES):
    ax.imshow(images[year])
    ax.set_title(f"{year} (Sentinel-2, true colour)", fontsize=14)
    ax.axis("off")
plt.suptitle("Hasdeo Arand – PEKB coal mine area, 16 km × 16 km", fontsize=15)
plt.tight_layout()
plt.savefig(FIG_DIR / "hasdeo_2019_vs_2025.png", dpi=120)
plt.show()

Zoom in on the mine: the middle 6 km.

In [ ]:
lo, hi = 500, 1100
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, year in zip(axes, SCENES):
    ax.imshow(images[year][lo:hi, lo:hi])
    ax.set_title(f"{year} – zoomed on the mine (6 km wide)", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 4 – Domain shift: do these look like EuroSAT?

Your answer to question 4 in notebook 03 was that European forest looks different from Indian forest. Let's measure it. Below: 6 EuroSAT **Forest** images next to 6 patches from the forested south of our study area.

In [ ]:
euro_forest = sorted((DATA_DIR / "Forest").glob("*.jpg"))[:6]
hasdeo_rows = [(20, c) for c in range(2, 24, 4)]          # patch positions in the southern forest

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, path in zip(axes[0], euro_forest):
    ax.imshow(Image.open(path)); ax.axis("off")
for ax, (r, c) in zip(axes[1], hasdeo_rows):
    ax.imshow(images[2019][r * PATCH:(r + 1) * PATCH, c * PATCH:(c + 1) * PATCH]); ax.axis("off")
axes[0, 0].set_title("EuroSAT Forest (Europe) →", loc="left")
axes[1, 0].set_title("Hasdeo forest (India, 2019) →", loc="left")
plt.tight_layout(); plt.show()

euro_mean = np.mean([np.array(Image.open(p)).reshape(-1, 3).mean(0) for p in sorted((DATA_DIR / "Forest").glob("*.jpg"))[:300]], axis=0)
hasdeo_mean = images[2019][1280:1600].reshape(-1, 3).mean(0)
print("Average colour (R, G, B)")
print("  EuroSAT forest:       ", euro_mean.round(0))
print("  Hasdeo southern area: ", hasdeo_mean.round(0))

Very different: EuroSAT forest is **bluish**, Hasdeo is **warm brown**. Two reasons:
1. **Different place and season.** This is dry tropical forest in February, with dry leaves and lots of exposed soil between trees. EuroSAT is temperate European forest.
2. **Different processing.** EuroSAT was made from images *without* atmospheric correction (L1C), which keeps a bluish haze. Ours are atmosphere-corrected (L2A), so the haze is removed.

**We tried fixing this** by re-colouring our images so their average colour and contrast match EuroSAT's (a common trick called colour matching). It made things *worse*: the model then called about half of all patches "Residential". Stretching the contrast exaggerated the texture of the trees. So we use the images as they are. That's a real result: an obvious fix didn't work. Worth one sentence in your paper's Discussion.

## Step 5 – Chop into patches

The model was trained on 64 × 64 images, so we cut each 1,600 × 1,600 image into a **25 × 25 grid** of 64 × 64 patches: 625 patches per year.

The `reshape` / `swapaxes` line does the cutting in one go: it reorganises the big grid of pixels into a stack of small squares, without any loops. Think of slicing a sheet cake into equal squares.

The picture shows the grid drawn over the 2019 image. Each square is one patch the model will classify.

In [ ]:
def to_patches(img):
    # (1600, 1600, 3) -> (25, 64, 25, 64, 3) -> (25, 25, 64, 64, 3) -> (625, 64, 64, 3)
    return img.reshape(N_PATCHES, PATCH, N_PATCHES, PATCH, 3).swapaxes(1, 2).reshape(-1, PATCH, PATCH, 3)

patches = {year: to_patches(images[year]) for year in SCENES}
print("Patches per year:", patches[2019].shape, "->  (patches, height, width, colours)")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(images[2019])
for k in range(N_PATCHES + 1):
    ax.axhline(k * PATCH, color="white", lw=0.4)
    ax.axvline(k * PATCH, color="white", lw=0.4)
ax.set_title("2019 image cut into 625 patches (each 640 m × 640 m)")
ax.axis("off")
plt.show()

## Step 6 – Classify every patch

Same steps as notebook 03: load your trained model, apply the same preparation (resize to 224, normalise), and get the 10 class probabilities for every patch. For each patch we keep the **predicted class** and the **confidence**.

Results are reshaped back into a 25 × 25 grid, so `pred[2019][row, col]` is the class of the patch at that spot.

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device).eval()

prepare = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

pred, conf, probs = {}, {}, {}
for year in SCENES:
    batch = torch.stack([prepare(Image.fromarray(p)) for p in patches[year]])
    with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16):
        p = torch.cat([torch.softmax(model(b.to(device)).float(), dim=1) for b in batch.split(128)]).cpu().numpy()
    probs[year] = p
    pred[year] = p.argmax(1).reshape(N_PATCHES, N_PATCHES)
    conf[year] = p.max(1).reshape(N_PATCHES, N_PATCHES)
    print(f"{year}: classified {len(p)} patches, average confidence {conf[year].mean():.1%}")

**Notice:** average confidence here is lower than the ~99% on EuroSAT's test set. The model is less sure on unfamiliar Indian images. That's the domain shift showing up as a number.

## Step 7 – Land cover maps

Each patch coloured by its predicted class. Left: the satellite image; right: what the model thinks is there.

In [ ]:
colours = plt.get_cmap("tab10")
legend = [Patch(color=colours(i), label=name) for i, name in enumerate(class_names)]

fig, axes = plt.subplots(2, 2, figsize=(15, 15))
for row, year in enumerate(SCENES):
    axes[row, 0].imshow(images[year])
    axes[row, 0].set_title(f"{year} – satellite image", fontsize=13)
    axes[row, 1].imshow(pred[year], cmap=colours, vmin=-0.5, vmax=9.5, interpolation="nearest")
    axes[row, 1].set_title(f"{year} – model's land cover map", fontsize=13)
for ax in axes.flat:
    ax.axis("off")
fig.legend(handles=legend, loc="lower center", ncol=5, fontsize=11)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig(FIG_DIR / "hasdeo_landcover_maps.png", dpi=110)
plt.show()

**What to look for:**
- Most of the area is **Forest** (orange), matching the dark areas in the image
- **Herbaceous Vegetation** (green) and **Permanent Crop** (pink) appear in the open, tan areas around villages and fields in the north
- The **mine** in 2025: look at what the model calls it. EuroSAT has no "mine" class, so it has to pick the closest thing it knows. The dark coal pit often comes out as **SeaLake**: dark and flat looks like water to it.

### How much of each class?

Counting patches per class, and converting to area (each patch = 0.41 km²).

In [ ]:
print(f"{'Class':22} {'2019':>6} {'2025':>6}   {'change':>7}   {'km² change':>10}")
for i, name in enumerate(class_names):
    a, b = (pred[2019] == i).sum(), (pred[2025] == i).sum()
    if a or b:
        print(f"{name:22} {a:6d} {b:6d}   {b - a:+7d}   {(b - a) * PATCH_KM2:+10.1f}")

Careful: these totals hide a lot. Patches can swap back and forth between classes, and many swaps cancel out in the totals. The next step looks at each patch individually.

## Step 8 – Change detection: where did forest disappear?

For every patch we ask: **was it Forest in 2019, and something else in 2025?** Those are our candidate **deforestation** patches.

From notebook 03 you know the model makes mistakes, so we also make a stricter version: only count a change if the model was **at least 90% confident on both dates**. Fewer detections, but more trustworthy ones.

`&` means "and": both conditions must be true for the same patch.

In [ ]:
FOREST = class_names.index("Forest")
CONFIDENT = 0.9

lost = (pred[2019] == FOREST) & (pred[2025] != FOREST)
lost_confident = lost & (conf[2019] >= CONFIDENT) & (conf[2025] >= CONFIDENT)
gained = (pred[2019] != FOREST) & (pred[2025] == FOREST)

print(f"Forest patches in 2019:            {(pred[2019] == FOREST).sum()}")
print(f"Forest -> something else:          {lost.sum():4d} patches  = {lost.sum() * PATCH_KM2:5.1f} km²")
print(f"   ...with ≥{CONFIDENT:.0%} confidence both years: {lost_confident.sum():4d} patches  = {lost_confident.sum() * PATCH_KM2:5.1f} km²")
print(f"Something else -> Forest:          {gained.sum():4d} patches  = {gained.sum() * PATCH_KM2:5.1f} km²")

**Wait, forest *gained*?** Forests don't grow back 640 m squares in six years. Those "gains" are almost certainly **model mistakes**: a patch that's half-forest, half-field can flip either way between dates. That gives you a rough sense of how many of the "losses" might be noise too. This is exactly the kind of honest check reviewers look for.

### The change map

Red boxes = forest lost (solid = confident, dashed = less confident). Blue boxes = forest "gained" (likely errors). Drawn over the 2025 image.

In [ ]:
def draw_boxes(ax, mask, colour, style="-"):
    for r, c in zip(*np.nonzero(mask)):
        ax.add_patch(Rectangle((c * PATCH, r * PATCH), PATCH, PATCH, fill=False, edgecolor=colour, lw=1.8, linestyle=style))

fig, axes = plt.subplots(1, 2, figsize=(17, 8.5))
axes[0].imshow(images[2019]); axes[0].set_title("2019", fontsize=14)
axes[1].imshow(images[2025]); axes[1].set_title("2025 – detected changes", fontsize=14)
draw_boxes(axes[1], lost & ~lost_confident, "red", "--")
draw_boxes(axes[1], lost_confident, "red")
draw_boxes(axes[1], gained, "deepskyblue")
for ax in axes:
    ax.axis("off")
axes[1].legend(handles=[Patch(edgecolor="red", fill=False, label=f"Forest lost, ≥{CONFIDENT:.0%} confident"),
                        Patch(edgecolor="red", fill=False, linestyle="--", label="Forest lost, less confident"),
                        Patch(edgecolor="deepskyblue", fill=False, label="Forest 'gained' (likely errors)")],
               loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "hasdeo_change_map.png", dpi=120)
plt.show()

Compare the red boxes with what *you* can see changed between the two images. Do they land on the mine and the new north-west clearing? Are there red boxes in places that look unchanged to you? Those would be false alarms.

**What we found when we checked this run by eye:**
- ✅ Most red boxes sit on **real change**: along the southern edge of the expanded mine pit, and on the **new clearing north-west** of the mine.
- ❌ The cluster of red boxes at the **very bottom** is a false alarm. Nothing changed there; it's dark forest in hill shadow, and the model called it **SeaLake**. And 3 of those were "≥90% confident"! So here, the confidence filter kept false alarms and threw away real mine expansion (where the model was *unsure*, because it has never seen a mine). The idea from notebook 03 (trust only confident predictions) **does not transfer** to this new place. That's a key finding for your paper.
- ❌ The many **blue boxes** are mostly in the open, scrubby north. The 2025 image is slightly darker and greener overall, so mixed scrub patches tipped over into "Forest". They show how much **noise** this simple method has: more false "gains" than real losses.

## Step 9 – What did the forest turn into?

For the patches that lost forest, which class did they become? This is where the "no mine class" problem shows up.

In [ ]:
became = pred[2025][lost]
print("Forest in 2019 became, in 2025:")
for i in np.argsort(-np.bincount(became, minlength=10)):
    n = (became == i).sum()
    if n:
        print(f"   {class_names[i]:22} {n:3d} patches")

## Step 10 – Look at the changed patches up close

Each pair: the same 640 m square in 2019 and 2025, with the model's label and confidence. Confident detections first.

Judge for yourself: is each one **real** forest loss, or a mistake?

In [ ]:
order = sorted(zip(*np.nonzero(lost)), key=lambda rc: -min(conf[2019][rc], conf[2025][rc]))[:12]

if order:
    fig, axes = plt.subplots(len(order), 2, figsize=(5, 2.6 * len(order)), squeeze=False)
    for (ax1, ax2), (r, c) in zip(axes, order):
        for ax, year in ((ax1, 2019), (ax2, 2025)):
            ax.imshow(images[year][r * PATCH:(r + 1) * PATCH, c * PATCH:(c + 1) * PATCH])
            ax.set_title(f"{year}: {class_names[pred[year][r, c]]} {conf[year][r, c]:.0%}", fontsize=8)
            ax.axis("off")
        ax1.set_ylabel(f"({r},{c})")
    plt.tight_layout(); plt.show()

## Step 11 – Save the results

We save the predictions, confidences and the map's location on Earth to `results/hasdeo_predictions.npz` (a small numpy file). **Notebook 05** loads it and compares our detections against **Global Forest Watch**, the independent dataset of tree cover loss.

In [ ]:
np.savez_compressed(
    RESULTS_DIR / "hasdeo_predictions.npz",
    pred_2019=pred[2019], pred_2025=pred[2025],
    conf_2019=conf[2019], conf_2025=conf[2025],
    probs_2019=probs[2019], probs_2025=probs[2025],
    crs=geo[2019]["crs"], transform=np.array(geo[2019]["transform"]),
    class_names=np.array(class_names), scenes=np.array(list(SCENES.values())),
)
print("Saved:", (RESULTS_DIR / "hasdeo_predictions.npz").resolve())
print("Figures:", [f.name for f in sorted(FIG_DIR.glob("hasdeo_*.png"))])

## Done!

You've built a deforestation map of a real place from free satellite data and your own model. 🎉

**But how good is it?** So far we only have the model's word for it. **Notebook 05** checks it against Global Forest Watch: how many of our detections they agree with, what we missed, and what we flagged that they didn't.

## Think about it (answer in chat)

1. Look at the change map. Did the red boxes land where *you* can see the mine grew? Point out one you think is a false alarm, and say why.
2. The model has no "mine" class. What did it call the mine, and why do you think it picked that?
3. Why does the "forest gained" number help you judge how trustworthy the "forest lost" number is?
4. A 640 m patch that's half forest, half mine: what should it be called? What does that mean for measuring deforestation with big patches?